# Design

> How comm-core is organized, and why.

In [ ]:
#| hide
import math

import numpy as np

from comm_core import *

## The idea

comm-core models the communication between agents that learn, independently of how they learn: MARL agents,
multi-agent world models, federated learning clients, or anything else where several parties exchange information.
It knows nothing about observations, policies or gradients. A payload is opaque: it is carried, delayed, corrupted
or lost, but never interpreted.

The work is split between the learning code and comm-core:

| the learning code decides | comm-core decides |
|---|---|
| **when** to communicate | **can** the message go (is there a link)? |
| **what** to send (the payload) | **when** does it go out (scheduling, medium access)? |
| **to whom** | **what** arrives (losses, bit errors, interference)? |
| | **what** did it cost (bits, latency, energy)? |

Because of this split, the same learning code runs over a perfect channel, a lossy one, a ray-traced radio
environment or (later) an external network simulator, by changing how the network is built, not the learning code.

## The pieces

```text
 learning code ── Network.send(Message) ──►  Network  ── Network.receive(node) ──► learning code
                                               │
          Topology          who can talk to whom (a Link per sender → receiver)
          Protocol          when each message goes out (Message → Transmission)
          Backend           how a step's transmissions are simulated
            └── Channel     what happens to one transmission (→ TransmissionResult)
          MetricsCollector  what it cost
```

Each abstraction answers one question:

| abstraction | question | implementations |
|---|---|---|
| `Topology` | Who can communicate with whom? | `FullMeshTopology`, `StarTopology`, `GraphTopology`, `DistanceTopology` |
| `Protocol` | When is a message transmitted? | `IdealProtocol`, `TDMAProtocol`, `SlottedALOHA` (`SlottedProtocol` to write others) |
| `CommunicationBackend` | How are the transmissions of a step simulated, together? | `AnalyticalBackend` (one at a time), `InterferenceBackend` (they interfere) |
| `Channel` | What happens to one transmission? | `IdentityChannel`, `AWGNChannel`, `SystemLevelChannel`, `LinkLevelChannel`, `SionnaRTChannel` |
| `MetricsCollector` | What did communication cost? | `CommunicationMetrics`: transmissions, losses, bits, latency, energy |
| `Network` | (ties them together) | |

A `Node` is passive: it does not send or receive by itself. The learning code acts on its behalf through the
`Network`.

## The data

Three records follow a message through the network:

* `Message`: what a node wants to say. It holds the sender, the receiver, the payload, a `type`, and
  `size_bits` (what the transmission costs).
* `Transmission`: one attempt to deliver a message. A message can take several attempts, e.g. retransmissions
  after a loss.
* `TransmissionResult`: what happened to one attempt. It records whether it succeeded, the payload as delivered,
  the bits, latency, energy, SNR, error rate and `metadata`.

The receiver gets the payload of the *result*, not the message's. Usually they are the same, but a link-level
channel delivers the payload with its bit errors.

## Life of a message

What `Network.step` does, in order:

1. `backend.step(time)`: time-varying parts move on.
2. `protocol.schedule(time)`: the messages allowed to go out in this step become transmissions. The others keep
   waiting in the protocol's queues.
3. `topology.get_link(...)`: the link of each transmission. It carries the bandwidth and the transmit power.
4. `backend.transmit_all([...])`: the step's transmissions, all at once, since they happen at the same time.
5. For each result, the metrics record it, and if it succeeded, the receiver's inbox gets the delivered payload.
6. `protocol.feedback(results)`: the acknowledgements, e.g. to retransmit what was lost.

`Network.send` only checks that a link exists and hands the message to the protocol; nothing is transmitted until
the next step. Here, three nodes take turns with TDMA: `b`'s message waits for `b`'s slot.

In [ ]:
nodes = ['a', 'b', 'c']
net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), TDMAProtocol(nodes), AnalyticalBackend(IdentityChannel()))
net.send(Message('a', 'b', payload='hello from a', size_bits=96))
net.send(Message('b', 'c', payload='hello from b', size_bits=96))
for step in range(2):
    results = net.step()
    print(f'step {step}:', [(r.transmission.sender, r.transmission.receiver, r.success) for r in results])
print('b got', [m.payload for m in net.receive('b')], '| c got', [m.payload for m in net.receive('c')])
print(net.metrics.summary())

step 0: [('a', 'b', True)]
step 1: [('b', 'c', True)]
b got ['hello from a'] | c got ['hello from b']
CommunicationMetrics(transmissions=2, successful=2, failed=0, bits_sent=192, bits_received=192, total_latency=0.0, total_energy=0.0)


## Time

A `Network` step advances the clock by `dt` (1 by default), and is one time slot. All transmissions of a step
happen at the same time. This is how protocols divide time (`SlottedProtocol`) and how interference arises
(`InterferenceBackend`).

There are two kinds of delay:

* **Queueing:** a message that waits in the protocol for a later step. The application sees it as a message
  arriving later.
* **Transmission:** `TransmissionResult.latency`, which is symbols / bandwidth when the link has a finite
  `bandwidth_hz`.

Applications driven by an environment, like `stable_marl.comm` in stable-marl, take one network step per
environment step.

## Fidelity: what happens to a transmission

The channels go from perfect to detailed. The radio channels follow the split of
[Sionna](https://nvlabs.github.io/sionna/) between system-level and link-level simulation:

| channel | models | use it for |
|---|---|---|
| `IdentityChannel` | Nothing is lost. | Upper bounds, debugging. |
| `AWGNChannel` | A fixed packet loss probability. | Simple ablations. |
| `SystemLevelChannel` | SNR from the link budget → packet error rate → intact or lost (a *PHY abstraction*). | Training, many agents: fast. |
| `LinkLevelChannel` | Every bit: payload → bits → symbols → `h √P x + n` → detected bits → payload with its bit errors. | Checking the abstraction, discrete messages that tolerate bit errors. |

The two radio channels share a base, `RadioChannel`, which computes the *link budget* of a transmission (`LinkBudget`):

```text
 Fading        → h           NoFading, RayleighFading, RayTracedFading (a Sionna RT table at the nodes' positions)
 PowerControl  → P, or not   FixedPower, ChannelInversion (outage beyond a budget), MessagePower (chosen by the sender)
 noise power   → SNR = P |h|² / noise power
```

They differ only in what they do with it. Moving between the levels therefore changes one class, and the levels
agree: the link level, with a CRC that drops corrupted packets, loses packets as often as the system level
predicts.

In [ ]:
def packet_loss(channel, steps=2000):
    net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), IdealProtocol(), AnalyticalBackend(channel))
    net.reset(seed=0)
    for _ in range(steps):
        net.send(Message('a', 'b', payload=np.arange(8) * 30, size_bits=64))   # 8 codebook indices: 64 bits
        net.step()
    return net.metrics.summary().packet_loss

print('SNR   predicted  system-level  link-level')
for snr_db in (4, 6, 8):
    noise = 10 ** (-snr_db / 10)
    print(f'{snr_db} dB  {packet_error_rate(snr_db, 64):9.3f}  {packet_loss(SystemLevelChannel(noise_power=noise)):12.3f}'
          f'  {packet_loss(LinkLevelChannel(noise_power=noise, drop_corrupted=True)):10.3f}')

SNR   predicted  system-level  link-level
4 dB      0.553         0.571       0.550
6 dB      0.142         0.156       0.143
8 dB      0.012         0.013       0.013


The link-level pieces (`IndexCodec`, `UniformQuantizer`, `BPSK`, `propagate`, `ChannelInversion.allocate`, ...)
are also plain vectorized functions on numpy arrays or torch tensors. Training code that needs whole batches of
links can use them directly, without the per-message objects of a `Network` (see `link`).

## Interference and medium access

A `Channel` sees one transmission at a time. Interference depends on all of a step's transmissions, so it belongs
in the backend, which gets them together (`CommunicationBackend.transmit_all`).

`InterferenceBackend` gives each transmission its SINR, `P |h|² / (noise + Σ P' |h'|²)`, where the sum runs over
the other senders and their paths `h'` come from the same fading. Radios are half-duplex: a node that transmits
cannot receive in the same step.

Avoiding collisions is then the protocol's job, as in a real medium access (MAC) layer. With four nodes on a line,
all of them with something to say at every step:

In [ ]:
xs = np.arange(4)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs])   # |h|^2 = 1e-6 / d^3
nodes = ['a', 'b', 'c', 'd']
table.set_positions({n: (i, 0) for i, n in enumerate(nodes)})

def delivered(protocol, steps=40):
    channel = SystemLevelChannel(fading=table, noise_power=1e-10, per_model='threshold', snr_threshold_db=10)
    net = Network([Node(n) for n in nodes], FullMeshTopology(nodes), protocol, InterferenceBackend(channel))
    net.reset(seed=0)
    for _ in range(steps):
        for s in nodes:
            for r in nodes:
                if s != r:
                    net.send(Message(s, r, None, size_bits=64))
        net.step()
    return net.metrics.summary().successful / (steps * len(nodes))        # per node and step, out of 3

for name, protocol in (('IdealProtocol', IdealProtocol()), ('TDMAProtocol', TDMAProtocol(nodes, queue_size=3)),
                       ('SlottedALOHA', SlottedALOHA(p=0.25, queue_size=3, seed=0))):
    print(f'{name:14s} {delivered(protocol):.2f} of 3 neighbours heard per step')

IdealProtocol  0.00 of 3 neighbours heard per step
TDMAProtocol   0.75 of 3 neighbours heard per step
SlottedALOHA   0.34 of 3 neighbours heard per step


`IdealProtocol` sends everything at once, so nobody hears: every node is transmitting. `TDMAProtocol` gives each
node its own slot, so there are no collisions, but every node waits for its turn. `SlottedALOHA` transmits at
random, and some transmissions collide.

## Moving nodes

Positions belong to the application (e.g. an environment), so comm-core is told about them; it never tracks them
itself:

* **Fading:** a fading that depends on positions has `set_positions`, e.g. `RayTracedFading`. The radio channels
  pass it on: `channel.set_positions({...})` before a step.
* **Topology:** for a range-limited topology, build a new `DistanceTopology` when the nodes move.

stable-marl's `CommunicationWrapper` does both at every environment step.

## Randomness and episodes

Every random component has its own generator: the channels, the fadings and `SlottedALOHA`. comm-core never
touches the global `random` or `numpy.random` state.

`Network.reset(seed)` starts a new episode. It empties the queues, the inboxes and the metrics, and reseeds the
protocol and the backend. The backend reseeds its channel, which reseeds its fading, each with a seed derived from
the one given. The same seed replays the same episode:

In [ ]:
def episode(net, seed):
    net.reset(seed)
    delivered = []
    for _ in range(50):
        net.send(Message('a', 'b', payload=0, size_bits=256))
        delivered += [r.success for r in net.step()]
    return delivered

net = Network([Node('a'), Node('b')], FullMeshTopology(['a', 'b']), SlottedALOHA(p=0.5),
              AnalyticalBackend(SystemLevelChannel(fading=RayleighFading(), noise_power=0.1)))
first, again, other = episode(net, 0), episode(net, 0), episode(net, 1)
print(f'seed 0 twice: the same episode ({first == again}); seed 1: another one ({first != other})')

seed 0 twice: the same episode (True); seed 1: another one (True)


## Extending

Each abstraction is a small base class. To add something, subclass the one that answers the question:

| to add | subclass | implement |
|---|---|---|
| a topology | `Topology` (or build a `GraphTopology` from `Link`s) | `neighbors`, `is_connected`, `get_link` |
| a medium access protocol | `SlottedProtocol` | `senders(slot, waiting)` |
| another kind of protocol | `Protocol` | `submit`, `schedule` (and `feedback`, `reset`) |
| a propagation model | `Fading` | `coefficient(transmission, link)` |
| a power control | `PowerControl` | `allocate(gain, noise_power, link, message)` |
| a modulation, a codec | `Modulation`, `Codec` | `modulate`/`demodulate`, `encode`/`decode` |
| a channel model | `Channel`, or `RadioChannel` to reuse the link budget | `transmit(transmission, link)` |
| an external simulator (ns-3, ...) | `CommunicationBackend` | `transmit`, `transmit_all` |

For example, a log-distance path loss computed from the nodes' positions plugs into both radio channels:

In [ ]:
class LogDistanceFading(Fading):
    "Path loss growing with distance: |h|^2 = gain_1m / d^exponent (no small-scale fading)."

    def __init__(self, exponent=3.0, gain_1m=1e-4):
        self.exponent, self.gain_1m, self.positions = exponent, gain_1m, {}

    def set_positions(self, positions):
        self.positions.update(positions)

    def coefficient(self, transmission, link):
        d = distance(self.positions[transmission.sender], self.positions[transmission.receiver])
        return math.inf if d == 0 else math.sqrt(self.gain_1m / d ** self.exponent)

channel = SystemLevelChannel(fading=LogDistanceFading(), power_control=ChannelInversion(10, max_power=0.1),
                             noise_power=1e-13)
channel.set_positions({'a': (0, 0), 'b': (10, 0), 'c': (300, 0)})
for receiver in ('b', 'c'):
    t = Transmission(Message('a', receiver, 'hi', size_bits=64), 'a', receiver, start_time=0.0)
    r = channel.transmit(t, Link('a', receiver))
    print(f'a -> {receiver}: success={r.success}, power={r.metadata["tx_power"]:.1e} W, outage={r.metadata.get("outage", False)}')

a -> b: success=True, power=1.0e-05 W, outage=False
a -> c: success=False, power=0.0e+00 W, outage=True


## Not (yet) in comm-core

* **A `CommunicationSystem` facade:** one object that builds the network from a few arguments, e.g.
  `topology='full_mesh', channel='awgn'`. The building blocks above would stay available for advanced use.
* **Broadcast as one transmission:** today, a message to several neighbours is several unicast transmissions, each
  with its own fate.
* **Listen-before-talk (CSMA)**, and TDMA frames computed from positions for spatial reuse.
* **Richer link-level models:** channel coding, other modulations, OFDM over the subcarriers of a ray-traced
  channel, MIMO.
* **Backends for external simulators** (Sionna link-level, ns-3, OpenAirInterface), at the backend level so
  that they get a whole step at once.